# EcoShield AI — Validation Threshold Tuning

Bu notebook Görev 2 ve Görev 3'te kaydedilen validation olasılıklarını kullanarak hafif ve ağır modellerin karar eşiklerini seçer.

Seçim kuralları:

- **Hafif model:** Recall en az `%90` olacak şekilde ağır modele yönlendirilen işlem oranını minimize et.
- **Ağır model:** Recall en az `%90` olacak şekilde F1 skorunu maksimize et.

Her modelin tüm benzersiz validation olasılıkları eşik adayı olarak değerlendirilir. Test verisi okunmaz, yüklenmez veya değerlendirilmez.

## 1. Ayarlar ve paket kontrolü

In [ ]:
MIN_LIGHT_RECALL = 0.90
MIN_HEAVY_RECALL = 0.90
CURVE_POINTS_PER_MODEL = 500

import importlib.util
import json
import os
import sys
import time
from pathlib import Path

required = {
    "matplotlib": "matplotlib", "numpy": "numpy", "pandas": "pandas",
    "psutil": "psutil", "pyarrow": "pyarrow", "sklearn": "scikit-learn",
    "tqdm": "tqdm",
}
missing = [pip for module, pip in required.items() if importlib.util.find_spec(module) is None]
if missing:
    raise ModuleNotFoundError("Eksik paketler: " + ", ".join(missing))

print("Threshold ayarları hazır.")
print("Hafif minimum recall:", MIN_LIGHT_RECALL)
print("Ağır minimum recall:", MIN_HEAVY_RECALL)

## 2. Importlar ve proje yolları

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil
from sklearn.metrics import average_precision_score, roc_auc_score
from tqdm.auto import tqdm

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "common_preprocessing.py").exists():
    candidate = NOTEBOOK_DIR / "notebooks"
    if (candidate / "common_preprocessing.py").exists():
        NOTEBOOK_DIR = candidate
if not (NOTEBOOK_DIR / "common_preprocessing.py").exists():
    raise FileNotFoundError("notebooks/common_preprocessing.py bulunamadı.")
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from common_preprocessing import build_project_paths, find_project_root

PROJECT_ROOT = find_project_root(Path.cwd())
PATHS = build_project_paths(PROJECT_ROOT)
PREDICTIONS_DIR = PATHS.outputs / "predictions"
FIGURES_DIR = PATHS.outputs / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

LIGHT_PREDICTIONS_PATH = PREDICTIONS_DIR / "light_models_validation_predictions.parquet"
HEAVY_PREDICTIONS_PATH = PREDICTIONS_DIR / "heavy_models_validation_predictions.parquet"

for path in [LIGHT_PREDICTIONS_PATH, HEAVY_PREDICTIONS_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"{path} bulunamadı. Önce ilgili model görevini çalıştırın.")

print("Proje kökü:", PROJECT_ROOT)
print("RAM:", f"{psutil.Process(os.getpid()).memory_info().rss / (1024**3):.2f} GB")

## 3. Validation olasılıklarını yükleme ve doğrulama

İki dosyanın aynı `TransactionID` sırasını ve aynı target değerlerini içerdiği doğrulanır.

In [ ]:
started = time.perf_counter()
light_predictions = pd.read_parquet(LIGHT_PREDICTIONS_PATH)
heavy_predictions = pd.read_parquet(HEAVY_PREDICTIONS_PATH)
print(f"Validation dosyaları {time.perf_counter() - started:.2f} sn içinde yüklendi.")

required_base_columns = {"TransactionID", "y_true"}
for name, frame in [("light", light_predictions), ("heavy", heavy_predictions)]:
    missing_columns = required_base_columns - set(frame.columns)
    if missing_columns:
        raise KeyError(f"{name} tahmin dosyasında eksik kolonlar: {missing_columns}")
    if frame["TransactionID"].duplicated().any():
        raise ValueError(f"{name} tahmin dosyasında tekrar eden TransactionID var.")
    if frame.isna().any().any():
        raise ValueError(f"{name} tahmin dosyasında eksik değer var.")

assert np.array_equal(
    light_predictions["TransactionID"].to_numpy(),
    heavy_predictions["TransactionID"].to_numpy(),
)
assert np.array_equal(
    light_predictions["y_true"].to_numpy(),
    heavy_predictions["y_true"].to_numpy(),
)
assert set(light_predictions["y_true"].unique()) == {0, 1}

print("Validation satırı:", len(light_predictions))
print("Fraud sayısı:", int(light_predictions["y_true"].sum()))
print("Light olasılık kolonları:", [c for c in light_predictions if c.endswith("_probability")])
print("Heavy olasılık kolonları:", [c for c in heavy_predictions if c.endswith("_probability")])

## 4. Tüm benzersiz threshold noktalarını hızlı hesaplama

Olasılıklar büyükten küçüğe bir kez sıralanır. Kümülatif TP/FP değerleri sayesinde her threshold için bütün validation dizisi tekrar tekrar taranmaz.

In [ ]:
def build_threshold_table(y_true, probabilities):
    y_true = np.asarray(y_true, dtype=np.int8)
    probabilities = np.asarray(probabilities, dtype=np.float64)
    if not np.isfinite(probabilities).all():
        raise ValueError("Olasılıklar NaN veya sonsuz değer içeriyor.")

    order = np.argsort(-probabilities, kind="mergesort")
    sorted_probabilities = probabilities[order]
    sorted_target = y_true[order]
    cumulative_tp = np.cumsum(sorted_target, dtype=np.int64)

    group_end_mask = np.r_[
        sorted_probabilities[:-1] != sorted_probabilities[1:],
        True,
    ]
    group_end_indices = np.flatnonzero(group_end_mask)
    thresholds = sorted_probabilities[group_end_indices]
    routed_count = group_end_indices + 1
    tp = cumulative_tp[group_end_indices]
    fp = routed_count - tp
    positive_count = int(y_true.sum())
    negative_count = len(y_true) - positive_count
    fn = positive_count - tp
    tn = negative_count - fp

    precision = np.divide(
        tp, tp + fp,
        out=np.zeros_like(tp, dtype=np.float64),
        where=(tp + fp) > 0,
    )
    recall = tp / positive_count
    f1 = np.divide(
        2 * precision * recall,
        precision + recall,
        out=np.zeros_like(precision),
        where=(precision + recall) > 0,
    )

    return pd.DataFrame({
        "threshold": thresholds,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "routed_count": routed_count,
        "routed_rate": routed_count / len(y_true),
        "heavy_call_reduction": 1.0 - (routed_count / len(y_true)),
        "tn": tn, "fp": fp, "fn": fn, "tp": tp,
    })


def downsample_curve(table, point_count=CURVE_POINTS_PER_MODEL):
    if len(table) <= point_count:
        return table.copy()
    indices = np.unique(np.linspace(0, len(table) - 1, point_count).astype(int))
    return table.iloc[indices].copy()


def select_light_threshold(table):
    eligible = table[table["recall"] >= MIN_LIGHT_RECALL].copy()
    if eligible.empty:
        selected = table.sort_values(
            ["recall", "routed_rate", "f1"],
            ascending=[False, True, False],
        ).iloc[0]
        reason = "Minimum recall sağlanamadı; en yüksek recall"
    else:
        selected = eligible.sort_values(
            ["routed_rate", "f1", "precision", "threshold"],
            ascending=[True, False, False, False],
        ).iloc[0]
        reason = f"Recall >= {MIN_LIGHT_RECALL:.2f} ile minimum routed rate"
    return selected, reason


def select_heavy_threshold(table):
    eligible = table[table["recall"] >= MIN_HEAVY_RECALL].copy()
    if eligible.empty:
        selected = table.sort_values(
            ["f1", "recall", "precision"],
            ascending=[False, False, False],
        ).iloc[0]
        reason = "Minimum recall sağlanamadı; en yüksek F1"
    else:
        selected = eligible.sort_values(
            ["f1", "precision", "recall", "threshold"],
            ascending=[False, False, False, False],
        ).iloc[0]
        reason = f"Recall >= {MIN_HEAVY_RECALL:.2f} şartında maksimum F1"
    return selected, reason

## 5. Hafif modellerin threshold analizi

In [ ]:
light_model_columns = {
    "LogisticRegression": "logistic_regression_probability",
    "DecisionTree": "decision_tree_probability",
    "SmallRandomForest": "small_random_forest_probability",
    "SmallLightGBM": "small_lightgbm_probability",
}

y_validation = light_predictions["y_true"].to_numpy(dtype=np.int8)
light_summary_rows = []
light_curve_frames = []

for model_name, probability_column in tqdm(
    light_model_columns.items(), desc="Hafif model threshold tuning", unit=" model"
):
    probabilities = light_predictions[probability_column].to_numpy()
    table = build_threshold_table(y_validation, probabilities)
    selected, reason = select_light_threshold(table)
    row = selected.to_dict()
    row.update({
        "model_name": model_name,
        "model_role": "light",
        "probability_column": probability_column,
        "roc_auc": float(roc_auc_score(y_validation, probabilities)),
        "pr_auc": float(average_precision_score(y_validation, probabilities)),
        "selection_reason": reason,
    })
    light_summary_rows.append(row)
    curve = downsample_curve(table)
    curve["model_name"] = model_name
    curve["model_role"] = "light"
    light_curve_frames.append(curve)
    print(
        f"{model_name}: threshold={row['threshold']:.6f}, "
        f"recall={row['recall']:.4f}, precision={row['precision']:.4f}, "
        f"routed={row['routed_rate']:.4f}"
    )

light_threshold_summary = pd.DataFrame(light_summary_rows).sort_values(
    ["routed_rate", "pr_auc"], ascending=[True, False]
).reset_index(drop=True)
light_threshold_curves = pd.concat(light_curve_frames, ignore_index=True)
display(light_threshold_summary)

## 6. Ağır modellerin threshold analizi

In [ ]:
heavy_model_columns = {
    "HeavyRandomForest": "heavy_random_forest_probability",
    "XGBoost": "xgboost_probability",
    "HeavyLightGBM": "heavy_lightgbm_probability",
    "HeavyCatBoost": "heavy_catboost_probability",
}

heavy_summary_rows = []
heavy_curve_frames = []

for model_name, probability_column in tqdm(
    heavy_model_columns.items(), desc="Ağır model threshold tuning", unit=" model"
):
    probabilities = heavy_predictions[probability_column].to_numpy()
    table = build_threshold_table(y_validation, probabilities)
    selected, reason = select_heavy_threshold(table)
    row = selected.to_dict()
    row.update({
        "model_name": model_name,
        "model_role": "heavy",
        "probability_column": probability_column,
        "roc_auc": float(roc_auc_score(y_validation, probabilities)),
        "pr_auc": float(average_precision_score(y_validation, probabilities)),
        "selection_reason": reason,
    })
    heavy_summary_rows.append(row)
    curve = downsample_curve(table)
    curve["model_name"] = model_name
    curve["model_role"] = "heavy"
    heavy_curve_frames.append(curve)
    print(
        f"{model_name}: threshold={row['threshold']:.6f}, "
        f"recall={row['recall']:.4f}, precision={row['precision']:.4f}, "
        f"f1={row['f1']:.4f}"
    )

heavy_threshold_summary = pd.DataFrame(heavy_summary_rows).sort_values(
    ["f1", "pr_auc"], ascending=False
).reset_index(drop=True)
heavy_threshold_curves = pd.concat(heavy_curve_frames, ignore_index=True)
display(heavy_threshold_summary)

## 7. Cascade için model ve threshold önerisi

Hafif tarafta minimum yönlendirme oranı, ağır tarafta maksimum F1 önceliklidir. Seçimler yalnızca validation sonuçlarına dayanır.

In [ ]:
selected_light = light_threshold_summary.sort_values(
    ["routed_rate", "pr_auc", "f1"], ascending=[True, False, False]
).iloc[0]
selected_heavy = heavy_threshold_summary.sort_values(
    ["f1", "pr_auc", "roc_auc"], ascending=False
).iloc[0]

selected_pipeline = pd.DataFrame([
    {
        "role": "light",
        "model_name": selected_light["model_name"],
        "probability_column": selected_light["probability_column"],
        "selected_threshold": selected_light["threshold"],
        "precision": selected_light["precision"],
        "recall": selected_light["recall"],
        "f1": selected_light["f1"],
        "routed_rate": selected_light["routed_rate"],
        "selection_reason": selected_light["selection_reason"],
    },
    {
        "role": "heavy",
        "model_name": selected_heavy["model_name"],
        "probability_column": selected_heavy["probability_column"],
        "selected_threshold": selected_heavy["threshold"],
        "precision": selected_heavy["precision"],
        "recall": selected_heavy["recall"],
        "f1": selected_heavy["f1"],
        "routed_rate": selected_heavy["routed_rate"],
        "selection_reason": selected_heavy["selection_reason"],
    },
])

display(selected_pipeline)

## 8. Threshold trade-off görselleri

In [ ]:
def plot_role_curves(curves, summary, role, output_name):
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    for model_name, model_curve in curves.groupby("model_name"):
        model_curve = model_curve.sort_values("threshold")
        axes[0].plot(model_curve["threshold"], model_curve["recall"], label=model_name)
        if role == "light":
            axes[1].plot(
                model_curve["threshold"], model_curve["routed_rate"], label=model_name
            )
        else:
            axes[1].plot(model_curve["threshold"], model_curve["f1"], label=model_name)

    axes[0].axhline(0.90, linestyle="--", color="black", label="Minimum recall")
    axes[0].set_title(f"{role.title()} — Recall / Threshold")
    axes[0].set_xlabel("Threshold")
    axes[0].set_ylabel("Recall")
    axes[1].set_title(
        f"{role.title()} — " + ("Routed Rate" if role == "light" else "F1")
    )
    axes[1].set_xlabel("Threshold")
    axes[1].set_ylabel("Routed Rate" if role == "light" else "F1")
    for axis in axes:
        axis.grid(alpha=0.25)
        axis.legend(fontsize=8)
    fig.tight_layout()
    output_path = FIGURES_DIR / output_name
    fig.savefig(output_path, dpi=170, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    return output_path


light_figure_path = plot_role_curves(
    light_threshold_curves, light_threshold_summary,
    "light", "light_threshold_tradeoffs.png",
)
heavy_figure_path = plot_role_curves(
    heavy_threshold_curves, heavy_threshold_summary,
    "heavy", "heavy_threshold_tradeoffs.png",
)

print("Light görsel:", light_figure_path.relative_to(PROJECT_ROOT))
print("Heavy görsel:", heavy_figure_path.relative_to(PROJECT_ROOT))

## 9. Sonuçları ve seçilen thresholdları kaydetme

In [ ]:
light_summary_path = PATHS.metrics / "light_threshold_tuning_summary.csv"
heavy_summary_path = PATHS.metrics / "heavy_threshold_tuning_summary.csv"
light_curves_path = PATHS.metrics / "light_threshold_curves.csv"
heavy_curves_path = PATHS.metrics / "heavy_threshold_curves.csv"
selected_pipeline_path = PATHS.metrics / "selected_pipeline_thresholds.csv"
metadata_path = PATHS.metadata / "threshold_tuning_metadata.json"

light_threshold_summary.to_csv(light_summary_path, index=False)
heavy_threshold_summary.to_csv(heavy_summary_path, index=False)
light_threshold_curves.to_csv(light_curves_path, index=False)
heavy_threshold_curves.to_csv(heavy_curves_path, index=False)
selected_pipeline.to_csv(selected_pipeline_path, index=False)

metadata = {
    "experiment_stage": "task_4_threshold_tuning",
    "split_version": "common_v2",
    "selection_dataset": "validation",
    "test_used": False,
    "minimum_light_recall": MIN_LIGHT_RECALL,
    "minimum_heavy_recall": MIN_HEAVY_RECALL,
    "light_selection_rule": "minimum routed rate subject to recall constraint",
    "heavy_selection_rule": "maximum F1 subject to recall constraint",
    "selected_light": selected_pipeline.iloc[0].to_dict(),
    "selected_heavy": selected_pipeline.iloc[1].to_dict(),
    "light_models": light_threshold_summary.to_dict(orient="records"),
    "heavy_models": heavy_threshold_summary.to_dict(orient="records"),
}
with metadata_path.open("w", encoding="utf-8") as file:
    json.dump(metadata, file, ensure_ascii=False, indent=2)

print("Kaydedilen dosyalar:")
for path in [
    light_summary_path, heavy_summary_path,
    light_curves_path, heavy_curves_path,
    selected_pipeline_path, metadata_path,
]:
    print(" -", path.relative_to(PROJECT_ROOT))

# Görev 4 tamamlanma koşulları

- Bütün hafif ve ağır modellerin validation olasılıkları tarandı.
- Test verisi okunmadı veya kullanılmadı.
- Hafif threshold minimum recall ve minimum routed rate kuralıyla seçildi.
- Ağır threshold minimum recall ve maksimum F1 kuralıyla seçildi.
- Her model için precision, recall, F1, FP/FN ve yönlendirme oranı kaydedildi.
- Seçilen hafif/ağır model ve thresholdlar Görev 5 cascade pipeline için kaydedildi.